In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv('spam.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [4]:
df = df.rename(columns={'Category': 'category', 'Message': 'message'})

In [5]:
df.head()

,category,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.groupby('category').describe()

message                                                            \
           count unique                                                top   
category                                                                     
ham         4825   4516                             Sorry, I'll call later   
spam         747    641  Please call our customer service representativ...   

               
         freq  
category       
ham        30  
spam        4

In [7]:
df['spam'] = df['category'].apply(lambda x: 1 if x == 'spam' else 0)

In [8]:
df

,category,message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will ü b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


In [9]:
X = df.drop(columns=['category', 'spam'], axis=1)
y = df['spam']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3)

In [10]:
print(type(X_train))
print(X_train.dtypes, '\n')  # if it's a DataFrame
print(X_train.values[:5])  # look at some examples

<class 'pandas.core.frame.DataFrame'>
message    object
dtype: object 

[['And pls pls drink plenty plenty water']
 ['Beerage?']
 ['Cbe is really good nowadays:)lot of shop and showrooms:)city is shaping good.']
 ['Hmph. Go head, big baller.']
 ['WELL DONE! Your 4* Costa Del Sol Holiday or £5000 await collection. Call 09050090044 Now toClaim. SAE, TCs, POBox334, Stockport, SK38xh, Cost£1.50/pm, Max10mins']]


In [11]:
X_train.head()

,message
4175,And pls pls drink plenty plenty water
783,Beerage?
2885,Cbe is really good nowadays:)lot of shop and s...
5000,"Hmph. Go head, big baller."
4299,WELL DONE! Your 4* Costa Del Sol Holiday or £5...


In [12]:
# Create a list of text documents from the 'message' column
X_train_texts = X_train['message'].tolist()

# Initialize and fit the CountVectorizer on training data
vec = TfidfVectorizer()
X_train_count = vec.fit_transform(X_train_texts).toarray()

# Train the Gaussian Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train_count, y_train)

# Prepare test data
emails = [
    'Hey mohan, can we watch football game tomorrow?',
    'Upto 20% discount on parking, exclusive offer just for you. Dont miss this reward!'
]

# Transform test data using the same vectorizer
emails_count = vec.transform(emails).toarray()

# Make predictions
predictions = nb_model.predict(emails_count)
print(predictions)

[0 0]


In [13]:
# Check the shapes of your test data
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}\n")

# Make sure X_test is properly formatted before transformation
if isinstance(X_test, pd.DataFrame) or isinstance(X_test, pd.Series):
    X_test_texts = X_test['message'].tolist() if 'message' in X_test.columns else X_test.tolist()
else:
    X_test_texts = X_test

# Transform the correctly formatted test texts
X_test_count = vec.transform(X_test_texts).toarray()

# Verify the shapes after transformation
print(f"X_test_count shape: {X_test_count.shape}")
print(f"y_test shape: {y_test.shape}")

X_test shape: (1672, 1)
y_test shape: (1672,)

X_test_count shape: (1672, 7092)
y_test shape: (1672,)


In [14]:
# Now score the model
score = nb_model.score(X_test_count, y_test)
print(f"Model accuracy: {score:.4f}")

Model accuracy: 0.8965


### Sklearn Pipepline

In [22]:
from sklearn.pipeline import Pipeline

In [15]:
# Check the shapes of your test data
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print()
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (3900, 1)
y_train shape: (3900,)

X_test shape: (1672, 1)
y_test shape: (1672,)


In [16]:
# Make sure X_test is properly formatted before transformation
if isinstance(X_train, pd.DataFrame) or isinstance(X_train, pd.Series):
    X_train_vec = X_train['message'].tolist() if 'message' in X_train.columns else X_train.tolist()
else:
    X_train_vec = X_train

# Transform the correctly formatted test texts
X_train_trans = vec.transform(X_train_vec).toarray()

# Verify the shapes after transformation
print(f"X_train_trans shape: {X_train_trans.shape}")
print(f"y_train shape: {y_train.shape}")

X_train_trans shape: (3900, 7092)
y_train shape: (3900,)


#### Not usig pipeline beacuse i already vectorized the data 

In [21]:
# If you want to use already vectorized data, skip the vectorizer in the pipeline
clf = MultinomialNB()
clf.fit(X_train_trans, y_train)

# Make sure X_test is also transformed the same way before scoring
X_test_trans = vec.transform(X_test_texts).toarray()  # Use your vectorizer 'vec'
score = clf.score(X_test_trans, y_test)
print(f"Accuracy: {score:.4f}")

Accuracy: 0.9557


Reshape your data either using array.reshape(-1, 1) if your data has a single feature or\
array.reshape(1, -1) if it contains a single sample.

In [22]:
# If you have access to the original vectorizer used during training:
emails_features = vec.transform(emails).toarray()  # This should create 7205 features
predictions = clf.predict(emails_features)

In [23]:
predictions

array([0, 0])